# Publications markdown generator for academicpages

Takes a set of bibtex of publications and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). 

The core python code is also in `pubsFromBibs.py`. 
Run either from the `markdown_generator` folder after replacing updating the publist dictionary with:
* bib file names
* specific venue keys based on your bib file preferences
* any specific pre-text for specific files
* Collection Name (future feature)

TODO: Make this work with other databases of citations, 
TODO: Merge this with the existing TSV parsing solution

In [8]:
from pybtex.database.input import bibtex
import pybtex.database.input.bibtex 
from time import strptime
import string
import html
import os
import re

In [9]:
publist = {
    "all": {
        "file": "suryanarayana_sankagiri_allpubs_abstracts.bib",  # put this next to the notebook
        "collection": {"name": "publications", "permalink": "/publication/"}
    }
}

In [10]:
ALLOWED_CATEGORIES = {"preprints", "journals", "conferences", "workshops", "books"}

def get_category(fields):
    # pybtex gives you raw string; you used keywords = {conferences} etc.
    kw = fields.get("keywords", "").strip().lower()
    # handle comma-separated keywords just in case
    parts = [p.strip() for p in kw.replace(";", ",").split(",") if p.strip()]
    for p in parts:
        if p in ALLOWED_CATEGORIES:
            return p
    # fallback: infer from entry type-ish signals if keyword missing
    return "preprints"  # default category

In [11]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [12]:
def get_venue(fields):
    # prefer booktitle (conf/workshop), else journal (journal-ish), else empty
    if "booktitle" in fields:
        return fields["booktitle"]
    if "journal" in fields:
        return fields["journal"]
    return ""

In [13]:
# Mappings for acronyms to full names
VENUE_MAPPING = {
    "ALT": "International Conference on Algorithmic Learning Theory (ALT)",
    "ICML": "International Conference on Machine Learning (ICML)",
    "UAI": "Conference on Uncertainty in Artificial Intelligence (UAI)",
    "SCC": "IEEE International Conference on Services Computing (SCC)",
    "NCC": "National Conference on Communications (NCC)",
    "ISIT": "IEEE International Symposium on Information Theory (ISIT)",
    "ISMIR": "International Society for Music Information Retrieval Conference (ISMIR)",
    "Blockchain": "IEEE International Conference on Blockchain",
    "ConsensusDay@CCS": "ConsensusDay Workshop at ACM CCS",
    # Add others here as needed
}

def clean_venue(venue_key):
    # Remove braces and backslashes if present
    v = venue_key.replace("{", "").replace("}", "").replace("\\", "")
    # Check if we have a prettier name in our mapping
    return VENUE_MAPPING.get(v, v)

In [14]:
for pubsource in publist:
    parser = bibtex.Parser()
    bibdata = parser.parse_file(publist[pubsource]["file"])

    #loop through the individual references in a given bibtex file
    for bib_id in bibdata.entries:
        #reset default date
        pub_year = "1900"
        pub_month = "01"
        pub_day = "01"
        
        b = bibdata.entries[bib_id].fields
        
        try:
            pub_year = f'{b["year"]}'

            #todo: this hack for month and day needs some cleanup
            if "month" in b.keys(): 
                if(len(b["month"])<3):
                    pub_month = "0"+b["month"]
                    pub_month = pub_month[-2:]
                elif(b["month"] not in range(12)):
                    tmnth = strptime(b["month"][:3],'%b').tm_mon   
                    pub_month = "{:02d}".format(tmnth) 
                else:
                    pub_month = str(b["month"])
            if "day" in b.keys(): 
                pub_day = str(b["day"])

                
            pub_date = pub_year+"-"+pub_month+"-"+pub_day
            
            #strip out {} as needed (some bibtex entries that maintain formatting)
            clean_title = b["title"].replace("{", "").replace("}","").replace("\\","").replace(" ","-")    

            url_slug = re.sub("\\[.*\\]|[^a-zA-Z0-9_-]", "", clean_title)
            url_slug = url_slug.replace("--","-")

            md_filename = (str(pub_date) + "-" + url_slug + ".md").replace("--","-")
            html_filename = (str(pub_date) + "-" + url_slug).replace("--","-")

            # --- Build Author List ---
            author_names = []
            if "author" in bibdata.entries[bib_id].persons:
                for author in bibdata.entries[bib_id].persons["author"]:
                    # Clean names
                    fn = " ".join(author.first_names).replace("{", "").replace("}", "")
                    ln = " ".join(author.last_names).replace("{", "").replace("}", "")
                    name = f"{fn} {ln}"
                    
                    # Bold your name
                    if "Sankagiri" in ln:
                        name = f"<b>{name}</b>"
                    
                    author_names.append(name)
            
            author_str = ", ".join(author_names)

            # --- Clean Venue Name ---
            raw_venue = get_venue(b).replace("{", "").replace("}", "").replace("\\", "")
            venue = clean_venue(raw_venue) # Uses the mapping function we defined earlier

            
            ## YAML variables
            md = "---\n"
            md += "title: \""   + html_escape(b["title"].replace("{", "").replace("}","").replace("\\","")) + '"\n'
            
            md += "authors: '" + html_escape(author_str) + "'\n"
            
            md += """collection: """ +  publist[pubsource]["collection"]["name"] + "\n"

            md += """\npermalink: """ + publist[pubsource]["collection"]["permalink"]  + html_filename

            md += "\ncategory: " + get_category(b)
            
            has_abstract = False
            if "abstract" in b.keys():
                if len(str(b["abstract"])) > 5:
                    has_abstract = True

            md += "\ndate: " + str(pub_date) 

            md += "\nvenue: '" + html_escape(venue) + "'"
            
            url = False
            if "url" in b.keys():
                if len(str(b["url"])) > 5:
                    md += "\npaperurl: '" + b["url"] + "'"
                    url = True

            # md += "\ncitation: '" + html_escape(citation) + "'"
            
            md += "\nexcerpt: ''"

            md += "\n---"

            
            ## Markdown description for individual page
            if has_abstract:
                md += "\n" + b["abstract"].strip() + "\n"

            if url:
                md += "\n[Access paper here](" + b["url"] + "){:target=\"_blank\"}\n" 
            # else:
            #     md += "\nUse [Google Scholar](https://scholar.google.com/scholar?q="+html.escape(clean_title.replace("-","+"))+"){:target=\"_blank\"} for full citation"

            md_filename = os.path.basename(md_filename)

            with open("../_publications/" + md_filename, 'w', encoding="utf-8") as f:
                f.write(md)
            print(f'SUCESSFULLY PARSED {bib_id}: \"', b["title"][:60],"..."*(len(b['title'])>60),"\"")
        # field may not exist for a reference
        except KeyError as e:
            print(f'WARNING Missing Expected Field {e} from entry {bib_id}: \"', b["title"][:30],"..."*(len(b['title'])>30),"\"")
            continue


SUCESSFULLY PARSED DBLP:conf/alt/SankagiriEFG26: " Recycling History: Efficient Recommendations from Contextual ... "
SUCESSFULLY PARSED DBLP:conf/alt/VillemaudSG26: " Ranking Items from Discrete Ratings: The Cost of Unknown Use ... "
SUCESSFULLY PARSED DBLP:conf/icml/SankagiriEG25: " Recommendations with Sparse Comparison Data: Provably Fast C ... "
SUCESSFULLY PARSED DBLP:conf/uai/CorreaSFG25: " Measuring {IIA} Violations in Similarity Choices with {B}aye ... "
SUCESSFULLY PARSED DBLP:journals/ton/SankagiriH25: " Pricing for Routing and Flow-Control in Payment Channel Netw ... "
SUCESSFULLY PARSED DBLP:journals/sigmetrics/SankagiriH24: " Pricing for Routing and Flow-Control in Payment Channel Netw ... "
SUCESSFULLY PARSED DBLP:conf/IEEEscc/SampathNDDS22: " Inventory Pooling using Deep Reinforcement Learning  "
SUCESSFULLY PARSED DBLP:conf/consensusday/AmeenSH22: " Blockchain Security when Messages are Lost  "
SUCESSFULLY PARSED DBLP:conf/fc/SankagiriWKV21: " Blockchain {CAP} Theorem 